In [1]:
from pathlib import Path
from collections import defaultdict

import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from datasets import Dataset

from transformers import (
    M2M100Tokenizer,
    M2M100ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup,
)

import sacrebleu

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [3]:
PROJECT_ROOT = Path(
    r"D:\dev\projects\fourlang_translation"
)

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "clean"
    / "en_uz"
    / "hplt"
    / "directional"
)

MODEL_OUTPUT_DIR = (
    PROJECT_ROOT
    / "models"
    / "m2m100_ft"
    / "en_uz_exp0"
)

RESULT_DIR = (
    PROJECT_ROOT
    / "results"
    / "m2m100_en_uz_exp0"
)

MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_FILE = DATA_DIR / "train.jsonl"
VALID_FILE = DATA_DIR / "validation.jsonl"
TEST_FILE = DATA_DIR / "test.jsonl"

print("TRAIN :", TRAIN_FILE)
print("VALID :", VALID_FILE)
print("TEST  :", TEST_FILE)
print()
print("MODEL :", MODEL_OUTPUT_DIR)
print("RESULT:", RESULT_DIR)

TRAIN : D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\directional\train.jsonl
VALID : D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\directional\validation.jsonl
TEST  : D:\dev\projects\fourlang_translation\data\clean\en_uz\hplt\directional\test.jsonl

MODEL : D:\dev\projects\fourlang_translation\models\m2m100_ft\en_uz_exp0
RESULT: D:\dev\projects\fourlang_translation\results\m2m100_en_uz_exp0


In [4]:
for file in [
    TRAIN_FILE,
    VALID_FILE,
    TEST_FILE,
]:
    print(
        file.name,
        "=>",
        file.exists()
    )

    assert file.exists(), (
        f"找不到文件: {file}"
    )

train.jsonl => True
validation.jsonl => True
test.jsonl => True


In [5]:
train_df = pd.read_json(
    TRAIN_FILE,
    lines=True,
)

valid_df = pd.read_json(
    VALID_FILE,
    lines=True,
)

test_df = pd.read_json(
    TEST_FILE,
    lines=True,
)

print("Train:", len(train_df))
print("Valid:", len(valid_df))
print("Test :", len(test_df))

Train: 17960
Valid: 1056
Test : 984


In [6]:
print(train_df.columns.tolist())

train_df.head(10)

['pair_id', 'src_lang', 'tgt_lang', 'src_text', 'tgt_text', 'direction', 'split']


,pair_id,src_lang,tgt_lang,src_text,tgt_text,direction,split
0,hplt_00000000,en,uz,"Good place, where you can wait for the, that w...","Yaxshi joy, qaerda kutish mumkin, bu hech qach...",en-uz,train
1,hplt_00000001,en,uz,- Faculty of Engineering,- Quruvchilik fakulteti,en-uz,train
2,hplt_00000002,en,uz,- They live mainly in woodland and forest regi...,- Ular asosan o'rmon va o'rmon mintaqalarida i...,en-uz,train
3,hplt_00000003,en,uz,"However, species can be found in desert regions.","Biroq, turlarni cho'l hududlarida topish mumkin.",en-uz,train
4,hplt_00000004,en,uz,- They have long tails and slender bodies.,- Ularning uzun quyruqlari va ingichka tanalar...,en-uz,train
5,hplt_00000005,en,uz,- Lacerticles feed mainly on insects.,- Lacerticles asosan hasharotlar bilan oziqlan...,en-uz,train
6,hplt_00000006,en,uz,Some species eat seeds.,Ba'zi turlari urug'larni eyishadi.,en-uz,train
7,hplt_00000007,en,uz,"AAAA is widely recognized by city, contea, sta...","AAAA keng shahar tomonidan e'tirof etilgan, tu...",en-uz,train
8,hplt_00000008,en,uz,Membership in this Association is open to any ...,Bu Assotsiatsiyaga a'zolik har qanday kishi uc...,en-uz,train
9,hplt_00000009,en,uz,- To promote social and cultural events in ord...,- Boshqa amerikaliklar bilan etnik merosini ba...,en-uz,train


In [7]:
print("Train directions:")

print(
    train_df[
        "direction"
    ].value_counts()
)

Train directions:
direction
en-uz    8980
uz-en    8980
Name: count, dtype: int64


In [8]:
train_pair_ids = set(
    train_df["pair_id"]
)

valid_pair_ids = set(
    valid_df["pair_id"]
)

test_pair_ids = set(
    test_df["pair_id"]
)

print(
    "Train ∩ Valid:",
    len(
        train_pair_ids
        &
        valid_pair_ids
    )
)

print(
    "Train ∩ Test:",
    len(
        train_pair_ids
        &
        test_pair_ids
    )
)

print(
    "Valid ∩ Test:",
    len(
        valid_pair_ids
        &
        test_pair_ids
    )
)

Train ∩ Valid: 0
Train ∩ Test: 0
Valid ∩ Test: 0


In [9]:
print("PyTorch:", torch.__version__)
print(
    "CUDA available:",
    torch.cuda.is_available()
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA version:",
        torch.version.cuda
    )

    total_memory = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / 1024**3
    )

    print(
        f"GPU Memory: {total_memory:.2f} GB"
    )

PyTorch: 2.13.0+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA version: 13.2
GPU Memory: 7.96 GB


In [10]:
USE_CUDA = torch.cuda.is_available()

USE_BF16 = (
    USE_CUDA
    and torch.cuda.is_bf16_supported()
)

USE_FP16 = (
    USE_CUDA
    and not USE_BF16
)

if USE_BF16:
    AMP_DTYPE = torch.bfloat16

elif USE_FP16:
    AMP_DTYPE = torch.float16

else:
    AMP_DTYPE = torch.float32

print("USE_CUDA:", USE_CUDA)
print("USE_BF16:", USE_BF16)
print("USE_FP16:", USE_FP16)
print("AMP_DTYPE:", AMP_DTYPE)

USE_CUDA: True
USE_BF16: True
USE_FP16: False
AMP_DTYPE: torch.bfloat16


In [11]:
BASE_MODEL = "facebook/m2m100_418M"

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

EPOCHS = 1

LEARNING_RATE = 2e-5

WEIGHT_DECAY = 0.01

WARMUP_RATIO = 0.05

GRAD_CLIP = 1.0

In [12]:
TRAIN_BATCH_SIZE = 2

VALID_BATCH_SIZE = 4

GRAD_ACCUM_STEPS = 8

In [13]:
tokenizer = (
    M2M100Tokenizer
    .from_pretrained(
        BASE_MODEL
    )
)

print("Tokenizer loaded.")

print(
    "en lang id:",
    tokenizer.get_lang_id("en")
)

print(
    "uz lang id:",
    tokenizer.get_lang_id("uz")
)

Tokenizer loaded.
en lang id: 128022
uz lang id: 128096


In [14]:
device = torch.device(
    "cuda"
    if USE_CUDA
    else "cpu"
)

print("Device:", device)

model = (
    M2M100ForConditionalGeneration
    .from_pretrained(
        BASE_MODEL
    )
)

model = model.to(device)

print("Model loaded.")

Device: cuda


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.94GB            

model.safetensors: downloading bytes:           |  0.00B            

Model loaded.


In [15]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Total parameters:"
    f" {total_params:,}"
)

print(
    f"Trainable parameters:"
    f" {trainable_params:,}"
)

print(
    f"Total:"
    f" {total_params / 1e6:.2f} M"
)

print(
    f"Trainable:"
    f" {trainable_params / 1e6:.2f} M"
)

Total parameters: 483,905,536
Trainable parameters: 483,905,536
Total: 483.91 M
Trainable: 483.91 M


In [16]:
model.gradient_checkpointing_enable()

model.config.use_cache = False

print(
    "Gradient checkpointing enabled."
)

Gradient checkpointing enabled.


In [17]:
UZ_BENCHMARK = [
    "Salom.",
    "Rahmat.",
    "Men talabaman.",
    "Bugun havo yaxshi.",
    "Men Toshkentda yashayman.",
    "Men ertaga ishga boraman.",
    "Men ertaga aeroportga boraman.",
    (
        "Men ertaga ertalab "
        "soat sakkizda "
        "aeroportga boraman."
    ),
]

In [18]:
UZ_BENCHMARK_REFS = [
    "Hello.",
    "Thank you.",
    "I am a student.",
    "The weather is good today.",
    "I live in Tashkent.",
    "I will go to work tomorrow.",
    "I will go to the airport tomorrow.",
    (
        "I will go to the airport "
        "at eight tomorrow morning."
    ),
]

In [19]:
@torch.inference_mode()
def translate_m2m(
    text,
    src_lang,
    tgt_lang,
    model_obj=None,
):
    if model_obj is None:
        model_obj = model

    model_obj.eval()

    tokenizer.src_lang = src_lang

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LENGTH,
    )

    encoded = {
        k: v.to(device)
        for k, v
        in encoded.items()
    }

    output = model_obj.generate(
        **encoded,
        forced_bos_token_id=(
            tokenizer.get_lang_id(
                tgt_lang
            )
        ),
        num_beams=5,
        max_new_tokens=128,
        early_stopping=True,
    )

    result = tokenizer.batch_decode(
        output,
        skip_special_tokens=True,
    )[0]

    return result

In [20]:
before_results = []

for source, reference in zip(
    UZ_BENCHMARK,
    UZ_BENCHMARK_REFS,
):

    prediction = translate_m2m(
        text=source,
        src_lang="uz",
        tgt_lang="en",
    )

    before_results.append({
        "source": source,
        "reference": reference,
        "prediction_before":
            prediction,
    })


before_df = pd.DataFrame(
    before_results
)

before_df

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

,source,reference,prediction_before
0,Salom.,Hello.,and Salom.
1,Rahmat.,Thank you.,and mercy.
2,Men talabaman.,I am a student.,But it is.
3,Bugun havo yaxshi.,The weather is good today.,by Havo Nashid.
4,Men Toshkentda yashayman.,I live in Tashkent.,You are in Yashima.
5,Men ertaga ishga boraman.,I will go to work tomorrow.,But it is Issue.
6,Men ertaga aeroportga boraman.,I will go to the airport tomorrow.,and the airport.
7,Men ertaga ertalab soat sakkizda aeroportga bo...,I will go to the airport at eight tomorrow mor...,The airport is located on the airport.


In [21]:
BEFORE_FILE = (
    RESULT_DIR
    / "uzbek_benchmark_before.csv"
)

before_df.to_csv(
    BEFORE_FILE,
    index=False,
    encoding="utf-8-sig",
)

print(BEFORE_FILE)

D:\dev\projects\fourlang_translation\results\m2m100_en_uz_exp0\uzbek_benchmark_before.csv


In [22]:
train_dataset = Dataset.from_pandas(
    train_df[
        [
            "pair_id",
            "src_lang",
            "tgt_lang",
            "src_text",
            "tgt_text",
            "direction",
        ]
    ],
    preserve_index=False,
)

valid_dataset = Dataset.from_pandas(
    valid_df[
        [
            "pair_id",
            "src_lang",
            "tgt_lang",
            "src_text",
            "tgt_text",
            "direction",
        ]
    ],
    preserve_index=False,
)

print(train_dataset)
print(valid_dataset)

Dataset({
    features: ['pair_id', 'src_lang', 'tgt_lang', 'src_text', 'tgt_text', 'direction'],
    num_rows: 17960
})
Dataset({
    features: ['pair_id', 'src_lang', 'tgt_lang', 'src_text', 'tgt_text', 'direction'],
    num_rows: 1056
})


In [23]:
def preprocess_example(example):

    src_lang = example[
        "src_lang"
    ]

    tgt_lang = example[
        "tgt_lang"
    ]

    tokenizer.src_lang = src_lang
    tokenizer.tgt_lang = tgt_lang

    encoded = tokenizer(
        example["src_text"],
        text_target=example[
            "tgt_text"
        ],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )

    return {
        "input_ids":
            encoded["input_ids"],

        "attention_mask":
            encoded[
                "attention_mask"
            ],

        "labels":
            encoded["labels"],
    }

In [24]:
tokenized_train = (
    train_dataset.map(
        preprocess_example,
        batched=False,
        remove_columns=
            train_dataset.column_names,
        desc="Tokenizing train",
    )
)

print(tokenized_train)

Tokenizing train:   0%|          | 0/17960 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 17960
})


In [25]:
tokenized_valid = (
    valid_dataset.map(
        preprocess_example,
        batched=False,
        remove_columns=
            valid_dataset.column_names,
        desc="Tokenizing validation",
    )
)

print(tokenized_valid)

Tokenizing validation:   0%|          | 0/1056 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1056
})


In [26]:
sample = tokenized_train[0]

print(
    "input_ids length:",
    len(sample["input_ids"])
)

print(
    "labels length:",
    len(sample["labels"])
)

print()
print("Input:")
print(
    tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=False,
    )
)

print()
print("Target:")
print(
    tokenizer.decode(
        sample["labels"],
        skip_special_tokens=False,
    )
)

input_ids length: 17
labels length: 23

Input:
__en__Good place, where you can wait for the, that will never happen.</s>

Target:
__uz__Yaxshi joy, qaerda kutish mumkin, bu hech qachon bo'lmaydi.</s>


In [27]:
for idx in range(
    min(
        10,
        len(tokenized_train)
    )
):
    item = tokenized_train[idx]

    print("=" * 70)

    print(
        "INPUT:",
        tokenizer.decode(
            item["input_ids"],
            skip_special_tokens=False,
        )
    )

    print(
        "TARGET:",
        tokenizer.decode(
            item["labels"],
            skip_special_tokens=False,
        )
    )

INPUT: __en__Good place, where you can wait for the, that will never happen.</s>
TARGET: __uz__Yaxshi joy, qaerda kutish mumkin, bu hech qachon bo'lmaydi.</s>
INPUT: __en__- Faculty of Engineering</s>
TARGET: __uz__- Quruvchilik fakulteti</s>
INPUT: __en__- They live mainly in woodland and forest regions.</s>
TARGET: __uz__- Ular asosan o'rmon va o'rmon mintaqalarida istiqomat qilishadi.</s>
INPUT: __en__However, species can be found in desert regions.</s>
TARGET: __uz__Biroq, turlarni cho'l hududlarida topish mumkin.</s>
INPUT: __en__- They have long tails and slender bodies.</s>
TARGET: __uz__- Ularning uzun quyruqlari va ingichka tanalari bor.</s>
INPUT: __en__- Lacerticles feed mainly on insects.</s>
TARGET: __uz__- Lacerticles asosan hasharotlar bilan oziqlanadi.</s>
INPUT: __en__Some species eat seeds.</s>
TARGET: __uz__Ba'zi turlari urug'larni eyishadi.</s>
INPUT: __en__AAAA is widely recognized by city, contea, state and federal officials as well as by civic organizations.</s>


In [28]:
data_collator = (
    DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100,
        return_tensors="pt",
    )
)

In [29]:
train_loader = DataLoader(
    tokenized_train,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=0,
)

valid_loader = DataLoader(
    tokenized_valid,
    batch_size=VALID_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=0,
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Valid batches:",
    len(valid_loader)
)

Train batches: 8980
Valid batches: 264


In [30]:
batch = next(
    iter(train_loader)
)

batch = {
    k: v.to(device)
    for k, v
    in batch.items()
}

model.train()

with torch.autocast(
    device_type=device.type,
    dtype=AMP_DTYPE,
    enabled=USE_CUDA,
):

    outputs = model(
        **batch
    )

print(
    "Loss:",
    outputs.loss.item()
)

Loss: 6.898604869842529


In [31]:
assert torch.isfinite(
    outputs.loss
), "Loss 出现 NaN / Inf"

print("Forward test passed.")

Forward test passed.


In [32]:
del batch
del outputs

gc.collect()

if USE_CUDA:
    torch.cuda.empty_cache()

In [33]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

In [34]:
optimizer_steps_per_epoch = (
    math.ceil(
        len(train_loader)
        /
        GRAD_ACCUM_STEPS
    )
)

total_training_steps = (
    optimizer_steps_per_epoch
    *
    EPOCHS
)

warmup_steps = int(
    total_training_steps
    *
    WARMUP_RATIO
)

print(
    "Optimizer steps / epoch:",
    optimizer_steps_per_epoch
)

print(
    "Total optimizer steps:",
    total_training_steps
)

print(
    "Warmup steps:",
    warmup_steps
)

Optimizer steps / epoch: 1123
Total optimizer steps: 1123
Warmup steps: 56


In [35]:
scheduler = (
    get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=
            total_training_steps,
    )
)

In [36]:
scaler = None

if USE_FP16:
    scaler = torch.amp.GradScaler(
        "cuda"
    )

print("Scaler:", scaler)

Scaler: None


In [37]:
@torch.inference_mode()
def evaluate_loss(
    model_obj,
    loader,
):

    model_obj.eval()

    losses = []

    progress = tqdm(
        loader,
        desc="Validation",
        leave=False,
    )

    for batch in progress:

        batch = {
            k: v.to(device)
            for k, v
            in batch.items()
        }

        with torch.autocast(
            device_type=device.type,
            dtype=AMP_DTYPE,
            enabled=USE_CUDA,
        ):

            outputs = model_obj(
                **batch
            )

        loss = (
            outputs.loss
            .detach()
            .float()
            .item()
        )

        losses.append(loss)

        progress.set_postfix(
            loss=f"{loss:.4f}"
        )

    return float(
        np.mean(losses)
    )

In [38]:
def train_one_epoch(
    model_obj,
    loader,
    optimizer_obj,
    scheduler_obj,
    epoch_index,
):

    model_obj.train()

    optimizer_obj.zero_grad(
        set_to_none=True
    )

    running_loss = 0.0

    optimizer_step_count = 0

    progress = tqdm(
        enumerate(loader),
        total=len(loader),
        desc=(
            f"Train Epoch "
            f"{epoch_index + 1}"
        ),
    )

    for batch_index, batch in progress:

        batch = {
            k: v.to(device)
            for k, v
            in batch.items()
        }

        with torch.autocast(
            device_type=device.type,
            dtype=AMP_DTYPE,
            enabled=USE_CUDA,
        ):

            outputs = model_obj(
                **batch
            )

            raw_loss = outputs.loss

            loss = (
                raw_loss
                /
                GRAD_ACCUM_STEPS
            )

        if not torch.isfinite(
            raw_loss
        ):

            raise RuntimeError(
                f"Non-finite loss:"
                f" {raw_loss.item()}"
            )

        if USE_FP16:

            scaler.scale(
                loss
            ).backward()

        else:

            loss.backward()

        running_loss += (
            raw_loss
            .detach()
            .float()
            .item()
        )

        should_step = (
            (
                batch_index + 1
            )
            %
            GRAD_ACCUM_STEPS
            ==
            0
            or
            (
                batch_index + 1
            )
            ==
            len(loader)
        )

        if should_step:

            if USE_FP16:

                scaler.unscale_(
                    optimizer_obj
                )

            torch.nn.utils.clip_grad_norm_(
                model_obj.parameters(),
                GRAD_CLIP,
            )

            if USE_FP16:

                scaler.step(
                    optimizer_obj
                )

                scaler.update()

            else:

                optimizer_obj.step()

            scheduler_obj.step()

            optimizer_obj.zero_grad(
                set_to_none=True
            )

            optimizer_step_count += 1

        average_loss = (
            running_loss
            /
            (
                batch_index
                + 1
            )
        )

        current_lr = (
            scheduler_obj
            .get_last_lr()[0]
        )

        progress.set_postfix(
            loss=(
                f"{raw_loss.item():.4f}"
            ),
            avg=(
                f"{average_loss:.4f}"
            ),
            lr=(
                f"{current_lr:.2e}"
            ),
        )

    return {
        "train_loss":
            running_loss
            /
            len(loader),

        "optimizer_steps":
            optimizer_step_count,
    }

In [39]:
initial_valid_loss = (
    evaluate_loss(
        model,
        valid_loader,
    )
)

print(
    "Initial Validation Loss:",
    initial_valid_loss
)

Validation:   0%|          | 0/264 [00:00<?, ?it/s]

Initial Validation Loss: 4.895721244089531


In [40]:
training_history = []

best_valid_loss = float("inf")

for epoch in range(EPOCHS):

    print()
    print("=" * 80)
    print(
        f"Epoch {epoch + 1}"
        f" / {EPOCHS}"
    )
    print("=" * 80)

    train_metrics = (
        train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            epoch,
        )
    )

    valid_loss = evaluate_loss(
        model,
        valid_loader,
    )

    record = {
        "epoch":
            epoch + 1,

        "train_loss":
            train_metrics[
                "train_loss"
            ],

        "valid_loss":
            valid_loss,

        "optimizer_steps":
            train_metrics[
                "optimizer_steps"
            ],
    }

    training_history.append(
        record
    )

    print()
    print(
        "Train Loss:",
        record["train_loss"]
    )

    print(
        "Valid Loss:",
        valid_loss
    )

    if valid_loss < best_valid_loss:

        best_valid_loss = (
            valid_loss
        )

        print()
        print(
            "New best model."
        )

        model.save_pretrained(
            MODEL_OUTPUT_DIR
        )

        tokenizer.save_pretrained(
            MODEL_OUTPUT_DIR
        )


Epoch 1 / 1


Train Epoch 1:   0%|          | 0/8980 [00:00<?, ?it/s]

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\WingYouther\.cache\huggingface\hub\models--facebook--m2m100_418M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Validation:   0%|          | 0/264 [00:00<?, ?it/s]


Train Loss: 3.163824327306652
Valid Loss: 2.3305618580092085

New best model.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [41]:
history_df = pd.DataFrame(
    training_history
)

history_df

,epoch,train_loss,valid_loss,optimizer_steps
0,1,3.163824,2.330562,1123


In [5]:
model = (
    M2M100ForConditionalGeneration
    .from_pretrained(
        MODEL_OUTPUT_DIR
    )
)

model = model.to(device)

model.eval()

NameError: name 'M2M100ForConditionalGeneration' is not defined

In [6]:
model.save_pretrained(
    MODEL_OUTPUT_DIR / "final"
)

tokenizer.save_pretrained(
    MODEL_OUTPUT_DIR / "final"
)

NameError: name 'model' is not defined

In [1]:
history_df.to_csv(
    RESULT_DIR
    / "training_history.csv",
    index=False,
    encoding="utf-8-sig",
)

NameError: name 'history_df' is not defined

In [2]:
print(
    "Initial valid loss:",
    initial_valid_loss
)

print(
    "Final valid loss:",
    training_history[-1][
        "valid_loss"
    ]
)

NameError: name 'initial_valid_loss' is not defined

In [3]:
del model

gc.collect()

if USE_CUDA:
    torch.cuda.empty_cache()

NameError: name 'model' is not defined

In [45]:
model = (
    M2M100ForConditionalGeneration
    .from_pretrained(
        MODEL_OUTPUT_DIR
    )
)

model = model.to(device)

model.eval()

model.config.use_cache = True

print(
    "Fine-tuned model reloaded."
)

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Fine-tuned model reloaded.


In [46]:
tokenizer = (
    M2M100Tokenizer
    .from_pretrained(
        MODEL_OUTPUT_DIR
    )
)

print(
    "Tokenizer reloaded."
)

Tokenizer reloaded.


In [47]:
after_results = []

for source, reference in zip(
    UZ_BENCHMARK,
    UZ_BENCHMARK_REFS,
):

    prediction = translate_m2m(
        text=source,
        src_lang="uz",
        tgt_lang="en",
    )

    after_results.append({
        "source":
            source,

        "reference":
            reference,

        "prediction_after":
            prediction,
    })


after_df = pd.DataFrame(
    after_results
)

after_df

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

,source,reference,prediction_after
0,Salom.,Hello.,Hi.
1,Rahmat.,Thank you.,Thank you.
2,Men talabaman.,I am a student.,I'm looking for.
3,Bugun havo yaxshi.,The weather is good today.,The weather today is good.
4,Men Toshkentda yashayman.,I live in Tashkent.,I'm living in the country.
5,Men ertaga ishga boraman.,I will go to work tomorrow.,I'm going to go back.
6,Men ertaga aeroportga boraman.,I will go to the airport tomorrow.,Then I go to the airport.
7,Men ertaga ertalab soat sakkizda aeroportga bo...,I will go to the airport at eight tomorrow mor...,I stay at the airport for a few hours later.


In [4]:
print(
    "Model files:"
)

for file in (
    MODEL_OUTPUT_DIR
    .iterdir()
):

    print(
        file.name
    )

Model files:


NameError: name 'MODEL_OUTPUT_DIR' is not defined

In [7]:
import gc
import json
import math
import random
import time

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    M2M100Tokenizer,
    M2M100ForConditionalGeneration,
)

In [8]:
PROJECT_ROOT = Path(
    r"D:\dev\projects\fourlang_translation"
)


MODEL_OUTPUT_DIR = (
    PROJECT_ROOT
    /
    "models"
    /
    "m2m100_ft"
    /
    "en_uz_exp0"
)

In [9]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [10]:
tokenizer = (
    M2M100Tokenizer
    .from_pretrained(
        MODEL_OUTPUT_DIR
    )
)

print("tokenizer loaded")

tokenizer loaded


In [11]:
model = (
    M2M100ForConditionalGeneration
    .from_pretrained(
        MODEL_OUTPUT_DIR
    )
)

model = model.to(device)

model.eval()

model.config.use_cache = True

print("model loaded")

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

model loaded


In [12]:
@torch.inference_mode()
def translate_m2m(
    text,
    src_lang,
    tgt_lang,
):

    tokenizer.src_lang = src_lang

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    encoded = {
        k:v.to(device)
        for k,v in encoded.items()
    }

    output = model.generate(
        **encoded,
        forced_bos_token_id=(
            tokenizer.get_lang_id(
                tgt_lang
            )
        ),
        num_beams=5,
        max_new_tokens=128,
    )

    result = tokenizer.batch_decode(
        output,
        skip_special_tokens=True,
    )[0]

    return result

In [13]:
text = "Men talabaman."

print(
    translate_m2m(
        text,
        "uz",
        "en"
    )
)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I'm looking for.


In [14]:
text = "Men talabaman."

print(
    translate_m2m(
        text,
        "uz",
        "en"
    )
)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I'm looking for.


In [16]:
train_df["direction"].value_counts()

NameError: name 'train_df' is not defined

In [17]:
translate_m2m(
"I am a student.",
"en",
"uz"
)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Men studentim.'

In [18]:
translate_m2m(
"Men talabaman.",
"uz",
"en"
)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"I'm looking for."